In [ ]:
# @title 1. Install New Gemini SDK and Setup
# Changed package to the new google-genai standard
!pip install -q -U google-genai

from google import genai
from google.colab import userdata
import os
import json
import time
import zipfile
from PIL import Image

# Setup API Key explicitly using your Colab Userdata Secrets
API_KEY = userdata.get('GEMINI_API_KEY')

# Initialize the new standard GenAI Client
client = genai.Client(api_key=API_KEY)

# Retaining your specified Gemini 3 Preview model
MODEL_ID = 'gemini-3-flash-preview'

print("✅ New GenAI SDK Installed and Client Configured.")

In [ ]:
# @title 2. Upload and Unzip Task 1 Data
from google.colab import files

print("Please upload 'Task1_Complete_Dataset.zip' generated from Task 1:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# Extract the zip file
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("task1_data")

# Define the new paths
IMG_FOLDER = "task1_data/page_images"
JSON_FOLDER = "task1_data/vector_data"

# Verify pairing
images = sorted([f for f in os.listdir(IMG_FOLDER) if f.endswith('.png')])
print(f"✅ Success! Unzipped {len(images)} images and their matching JSON files.")

In [ ]:
# @title 3. Run Automated Batch Analysis (Final Universal Framework)
import re
import time
import os
import json
from PIL import Image

# --- 1. FUNCTION: Natural Sorting (Ensures Sheet 2 comes before Sheet 10) ---
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

# --- 2. FUNCTION: JSON Pruning (Solves the "400: Token Limit" Error) ---
def prune_v_data(v_data, max_items=12000):
    """If the JSON is too dense, it prioritizes text and thins out geometry lines."""
    pruned = v_data.copy()

    # Priority 1: Preserve all text (Vector and OCR)
    text_count = len(v_data.get('text', [])) + len(v_data.get('raster_ocr', []))
    geom_list = v_data.get('detected_lines', [])

    # Priority 2: If the total object count is too high, thin out the wall/contour lines
    if text_count + len(geom_list) > max_items:
        print(f"--- ✂️ Pruning geometry for token safety (Total items: {text_count + len(geom_list)})")
        # Keep every 5th line to reduce token weight while keeping the drawing's shape
        pruned['detected_lines'] = geom_list[::5]
        pruned['note'] = "Geometry thinned to stay within token limits."

    return pruned

# --- 3. THE UNIVERSAL MASTER PROMPT ---
MASTER_PROMPT = """
You are an expert GDOT (Georgia Department of Transportation) structural engineering analyst with deep knowledge of AASHTO standards, the GDOT Plan Development Process (PDP), and standard structural/civil drawing conventions.

You are analyzing a multi-layer "Digital Twin" document payload of a BRIDGE & CIVIL STRUCTURE Cover Sheet consisting of:
1. PNG Raster Image — Visual context and layout of the cover sheet.
2. Vector JSON — Direct, selectable text extracted with associated bounding box X/Y coordinates.
3. OCR JSON — Text recovered computationally from flattened or rasterized document layers (logos, seals, stamps).

Your task is to cross-reference ALL three data sources to produce a highly accurate, structured extraction of this bridge project cover sheet. Do not assume facts not explicitly supported by the data layers. If data is absent from all sources, explicitly flag it as [NOT FOUND]. For every single extracted property, you must provide a corresponding source token mapping exactly where the data was located ("vector" | "ocr" | "visual" | "NOT FOUND").

---

## 1. SHEET IDENTIFICATION
- Sheet number
- Drawing number / project code designation
- Document Type: Cover Sheet / Title Sheet
- Verified Project Engineering Domain: Bridge Project

## 2. PROJECT IDENTIFICATION
- Full project title and confirmed active Project P.I. Number (e.g., 331910 / 343455)
- County location and designated State Route (SR) / US Route / Road name
- Assigned GDOT District number
- Federal Aid Project Number (if explicitly present)
- Letting date or advertisement date
- Programmed fiscal year

## 3. RESPONSIBLE PARTIES
- Design firm, consulting engineering agency, or GDOT office of record
- Engineer of Record (EOR) full name and structural PE seal visibility status
- GDOT reviewer / approver name, title, and signature block status

## 4. QUANTITATIVE LOCATION DATA
- Specific State Route (SR) or US Route number
- Project Begin/End station boundaries or mileposts (e.g., STA XX+XX)
- Explicit GPS coordinates or county map reference vectors printed on the sheet

## 5. TITLE BLOCK VERIFICATION
- Active revision index table contents (dates, entities requesting updates, modified sheet logs)

## 6. FLAGS & ANOMALIES
- Missing or hidden PE stamp/signature
- Discrepancies where fields are visually clear in the image but missing from vector/OCR JSON data
- Skewed, rotated, or vertical text vectors, project logos, or state shields that raw computational OCR engines missed

## 7. EXPERT RISK & COORDINATION INSIGHTS
- Identify 3-5 specific construction risks if there are any visible or implied on this sheet.
- Target Examples: Excessive structural spans indicating unfeasible pier placements, utility/railroad right-of-way encroachments (e.g., CSX track interfaces), challenging hydraulic vertical clearance over historical stream floodplains, staging constraints for crane placement during girder installation, or discrepancies between structural station limits and civil roadway approach baselines.

---

## REQUIRED SUMMARY TABLE
- Create an Extraction Summary Table matching columns: Property Name | Extracted Value | Source Layer Used (Vector/OCR/Visual)
"""

# --- 4. CONFIGURATION & CHECKPOINT SYSTEM ---
RESULTS_FILE = "Final_Project_Full_Analysis.md"
processed_sheets = []

if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        existing_content = f.read()
        processed_sheets = re.findall(r"# ANALYSIS: (.*?)\n", existing_content)

images = [f for f in os.listdir(IMG_FOLDER) if f.endswith('.png')]
images.sort(key=natural_sort_key)


# =================================================================
# ⚙️ STEP 1: INITIALIZED COBA LEDGER COUNTERS (Placed Correctly!)
# =================================================================
total_input_tokens_sent = 0
total_output_tokens_received = 0
total_sheets_processed = 0

# Gemini 3 Flash Preview Standard Tier Production Tariffs
TRUE_INPUT_PRICE_PER_1M = 0.50   # $0.50 per 1,000,000 tokens
TRUE_OUTPUT_PRICE_PER_1M = 3.00  # $3.00 per 1,000,000 tokens

print(f"📊 Progress: {len(processed_sheets)} / {len(images)} sheets already in report.")
print(f"🚀 Processing missing sheets...")


# --- 5. EXECUTION LOOP ---
for img_file in images:
    if img_file in processed_sheets:
        continue

    json_file = img_file.replace('.png', '.json')
    img_path = os.path.join(IMG_FOLDER, img_file)
    json_path = os.path.join(JSON_FOLDER, json_file)

    if os.path.exists(json_path):
        img_obj = Image.open(img_path)
        with open(json_path, 'r') as f:
            v_data = json.load(f)

        v_data_to_send = prune_v_data(v_data)

        success = False
        attempts = 0
        while not success and attempts < 3:
            try:
                print(f"Processing: {img_file} (Attempt {attempts + 1})...")
                json_payload = json.dumps(v_data_to_send)

                # API Call with Grounding updated to new SDK standard syntax
                response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=[
                       f"{MASTER_PROMPT}\n\nGROUNDING DATA (JSON):\n{json_payload}",
                       img_obj
                    ]
                )

                # Record results
                report_section = f"# ANALYSIS: {img_file}\n\n{response.text}\n\n---\n"

                # Immediate Save (Append Mode)
                with open(RESULTS_FILE, "a") as f:
                    f.write(report_section)


                # =================================================================
                # 🛠️ STEP 2: EXTRACT METRICS STRAIGHT FROM GOOGLE'S SERVER
                # =================================================================
                success = True
                total_sheets_processed += 1

                # Pull exact token balances returned in the response headers
                if hasattr(response, 'usage_metadata') and response.usage_metadata:
                    total_input_tokens_sent += response.usage_metadata.prompt_token_count
                    total_output_tokens_received += response.usage_metadata.candidates_token_count
                else:
                    # Absolute emergency network fallback if metadata drops out
                    fallback_text_size = len(json_payload + MASTER_PROMPT) // 4
                    total_input_tokens_sent += fallback_text_size + 258
                    total_output_tokens_received += 3500

                # Running cost ledger updates printed every 5 sheets
                if total_sheets_processed % 5 == 0:
                    run_in_cost = (total_input_tokens_sent / 1_000_000) * TRUE_INPUT_PRICE_PER_1M
                    run_out_cost = (total_output_tokens_received / 1_000_000) * TRUE_OUTPUT_PRICE_PER_1M
                    print(f"   💰 Running Cost Ledger: ~${(run_in_cost + run_out_cost):.3f} for {total_sheets_processed} sheets")

                time.sleep(8)

            except Exception as e:
                error_msg = str(e)
                if "429" in error_msg:
                    print(f"⚠️ Quota Limit Hit. Waiting 90 seconds...")
                    time.sleep(90)
                    attempts += 1
                elif "400" in error_msg:
                    print(f"❌ Token limit exceeded. Applying aggressive pruning...")
                    v_data_to_send['detected_lines'] = []
                    attempts += 1
                else:
                    print(f"❌ Error on {img_file}: {e}")
                    break


# =================================================================
# 📈 STEP 3: REVISED TRUE PRODUCTION ACCOUNTING AUDIT
# =================================================================
final_input_cost = (total_input_tokens_sent / 1_000_000) * TRUE_INPUT_PRICE_PER_1M
final_output_cost = (total_output_tokens_received / 1_000_000) * TRUE_OUTPUT_PRICE_PER_1M
true_total_calculated_cost = final_input_cost + final_output_cost

print(f"\n" + "="*60)
print(f"✅ BATCH OPERATION AUDIT COMPLETE")
print(f"  Sheets successfully written:    {total_sheets_processed}")
print(f"  True Input Context Tokens:      {total_input_tokens_sent:,}")
print(f"  True Output Generated Tokens:   {total_output_tokens_received:,}")
print(f"  ------------------------------------------------------------")
print(f"  Calculated Input Layer Cost:    ${final_input_cost:.4f}")
print(f"  Calculated Output Layer Cost:   ${final_output_cost:.4f}")
print(f"  🚨 TRUE CONSOLE LEDGER TOTAL:   ${true_total_calculated_cost:.3f}")
print(f"  Reference Target (Optimized):   ~${(total_sheets_processed * 0.004):.3f}")
print(f"="*60)

print(f"\n✅ FULL PROJECT AUDIT COMPLETE. Report saved as: {RESULTS_FILE}")

In [ ]:
# @title 4. View Analysis Report Directly
from IPython.display import Markdown, display

# Check if the results file exists before trying to read it
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        full_report = f.read()

    # Display the content using Markdown formatting
    display(Markdown("## 📋 ARCHITECTURAL ANALYSIS REPORT"))
    display(Markdown(full_report))
else:
    print(f"⚠️ Error: The file '{RESULTS_FILE}' was not found. Please run the analysis cell first.")